# Người 3 — IPN Hand: template, instance graph, V4 và so sánh baseline
Chọn **Runtime → Change runtime type → T4 GPU**. Chạy lần lượt từng cell.

Đầu vào trên Drive: `ipn-extract-data/train.txt`, `val.txt`, `test.txt`, `manifest.csv`, JSON trong `keypoints/{split}/{label}/{subject}/`, tensor trong `keypoint_tensor_cache/T64/{split}/{label}/{subject}/`. Nhận checkpoint **best theo validation** từ người 2 nếu muốn so sánh baseline.

Notebook dùng JSON gốc để dựng graph: cache 64 frame đã mất cờ valid và thay đổi trục thời gian, nên không thay thế JSON ở bước này. Tensor .pt dùng cho nhánh neural. Không cần cài RTMW/MMPose hoặc trích lại video.

Các template bên dưới là **bản nháp có giới hạn**, không phải template đã nghiệm thu. Xem clip train/val, thống kê concept và matcher rồi chỉnh YAML. Không chỉnh template/hyperparameter từ test. Notebook không tự chạy test khi Run all; cell test có công tắc riêng. Kết quả ghi vào thư mục run riêng trên Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO = Path('/content/ipn-person3-repo')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = '53ed536'  # phiên bản code đã dùng để viết notebook
RUN_NAME = 'person3_v4_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# Khi reconnect: đặt RUN_NAME về tên run cũ và RESUME=True.
RESUME = False
BASELINE_CHECKPOINT = None  # ví dụ DRIVE_ROOT / 'person2_baseline/best.pt'
BASELINE_INPUT_HASHES = None  # JSON hash train.txt/val.txt/test.txt/manifest.csv do người 2 bàn giao
RUN = DRIVE_ROOT / 'person3_runs' / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
print('Output:', RUN)


In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_REF], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'pyyaml', 'tqdm', 'scikit-learn', 'matplotlib', 'networkx'], check=True)
import torch, pandas as pd, numpy as np
from tqdm.auto import tqdm
from IPython.display import display, Video
assert torch.cuda.is_available(), 'Chọn GPU runtime Colab rồi chạy lại.'
DEVICE = 'cuda:0'
CODE = REPO / 'final/code'
sys.path.insert(0, str(CODE))
os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
(RUN / 'environment.txt').write_text(
    f'commit={commit}\npython={sys.version}\ntorch={torch.__version__}\ngpu={torch.cuda.get_device_name(0)}\n'
    + subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print(torch.cuda.get_device_name(0), commit)


## Kiểm tra dữ liệu và sửa đường dẫn cho Colab
Không đổi split hoặc label. Notebook tạo bản split dùng đường dẫn tuyệt đối tới JSON trên Drive; việc này tránh lỗi resolver với tên subject IPN. Kiểm tra toàn bộ tensor, nhãn và subject leakage trước khi train. Chỉ nạp .pt/checkpoint do nhóm mình tạo.


In [ ]:
LABELS = ['B0A', 'B0B'] + [f'G{i:02d}' for i in range(1, 12)]
LABEL_IDS = {name: i for i, name in enumerate(LABELS)}
KEYPOINT_ROOT = DRIVE_ROOT / 'keypoints'
CACHE_ROOT = DRIVE_ROOT / 'keypoint_tensor_cache'  # không thêm T64 ở đây
SPLIT_ROOT = RUN / 'splits'
SPLIT_ROOT.mkdir(exist_ok=True)
rows = []
for split in ('train', 'val', 'test'):
    lines = []
    source_file = DRIVE_ROOT / f'{split}.txt'
    for line_no, line in enumerate(source_file.read_text(encoding='utf-8-sig').splitlines(), 1):
        if not line.strip():
            continue
        parts = line.rsplit(maxsplit=2)
        assert len(parts) == 3, (source_file, line_no)
        source, middle, label_text = parts
        pieces = source.replace('\\', '/').split('/')
        assert split in pieces, source
        rel = Path(*pieces[pieces.index(split):])
        label = rel.parts[1]
        assert LABEL_IDS[label] == int(label_text), (source, label_text)
        raw = KEYPOINT_ROOT / rel
        pt = CACHE_ROOT / 'T64' / rel.with_suffix('.pt')
        assert raw.is_file() and pt.is_file(), f'Thiếu JSON/PT: {raw}, {pt}'
        rows.append(dict(split=split, label=label, label_id=int(label_text),
                         source=source, relative=rel.as_posix(), raw=str(raw), pt=str(pt)))
        lines.append(f'{raw.as_posix()} {middle} {label_text}')
    assert lines, f'Split rỗng: {split}'
    (SPLIT_ROOT / f'{split}.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
samples = pd.DataFrame(rows)
assert not samples['relative'].duplicated().any(), 'Trùng sample.'
assert not samples['raw'].duplicated().any()
for split in ('train', 'val', 'test'):
    assert set(samples.loc[samples.split == split, 'label']) == set(LABELS)
manifest = pd.read_csv(DRIVE_ROOT / 'manifest.csv')
assert {'subject', 'split', 'keypoint_path', 'label_id'}.issubset(manifest.columns), 'Cần manifest người 1 có subject/split/keypoint_path/label_id.'
def relative_key(path, split):
    parts = str(path).replace('\\', '/').split('/')
    return '/'.join(parts[parts.index(split):])
manifest['relative'] = [relative_key(p, s) for p,s in zip(manifest.keypoint_path, manifest.split)]
assert not manifest.relative.duplicated().any(), 'Manifest trùng clip.'
linked = samples.merge(manifest[['relative','subject','label_id']], on='relative', how='left', validate='one_to_one', suffixes=('', '_manifest'))
assert linked.subject.notna().all(), 'Split chứa clip không có trong manifest.'
assert (linked.label_id == linked.label_id_manifest).all(), 'Nhãn split khác manifest.'
# Manifest có thể còn mẫu đã loại; kiểm tra leakage trên đúng tập mẫu hiện dùng.
manifest = linked
assert manifest[['subject', 'split']].notna().all().all()
subjects = {s: set(manifest.loc[manifest.split == s, 'subject']) for s in ('train', 'val', 'test')}
assert all(subjects.values())
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not subjects[a] & subjects[b], f'Subject leakage: {a}, {b}'
from src_neurosymbolic.pose.normalize_keypoints import normalize_video
from src_neurosymbolic.models.keypoint_dataset import keypoints_to_tensor
data_digest = hashlib.sha256()
for row in tqdm(rows, desc='Đối chiếu JSON và tensor'):
    x = torch.load(row['pt'], map_location='cpu', weights_only=True)
    assert isinstance(x, torch.Tensor) and x.shape == (64, 52, 3), row['pt']
    assert x.dtype == torch.float32 and torch.isfinite(x).all() and x.abs().sum() > 0, row['pt']
    assert ((x[..., 2] >= 0) & (x[..., 2] <= 1)).all(), row['pt']
    expected_tensor = keypoints_to_tensor(normalize_video(Path(row['raw'])), 64)
    assert torch.allclose(x, expected_tensor, atol=1e-6, rtol=1e-5), f'Cache khác JSON/pipeline: {row["pt"]}. Tạo lại cache hoặc thống nhất preprocessing với người 1.'
    data_digest.update(row['relative'].encode())
    data_digest.update(hashlib.sha256(Path(row['raw']).read_bytes()).digest())
    data_digest.update(hashlib.sha256(Path(row['pt']).read_bytes()).digest())
samples.to_csv(RUN / 'samples.csv', index=False)
fingerprints = {f: hashlib.sha256((DRIVE_ROOT / f).read_bytes()).hexdigest()
                for f in ['train.txt', 'val.txt', 'test.txt', 'manifest.csv']}
(RUN / 'input_hashes.json').write_text(json.dumps(fingerprints, indent=2))
display(pd.crosstab(samples.label, samples.split).reindex(LABELS))


## Dựng graph từ JSON và thống kê concept trên train/val
Dùng đúng normalize → predicates → events → instance graph của repo. Giữ cờ valid từ JSON.
Mỗi run có thư mục graph riêng để tránh dùng nhầm graph sinh bằng tham số cũ.
Chưa xem hoặc thống kê nội dung test ở bước thiết kế template.


In [ ]:
from src_neurosymbolic.pose.normalize_keypoints import normalize_video
from src_neurosymbolic.predicates.extract_predicates import extract_predicates
from src_neurosymbolic.events.build_events import build_events
from src_neurosymbolic.kg.build_instance_graph import build_instance_graph
from src_neurosymbolic.models.keypoint_dataset import KeypointSequenceDataset
from collections import Counter

GRAPH_ROOT = RUN / 'instance_graphs'
ARTIFACT_ROOT = RUN / 'symbolic_examples'
def make_graphs(split):
    selected = samples[samples.split == split]
    for row in tqdm(selected.to_dict('records'), desc=f'Graph {split}'):
        out = GRAPH_ROOT / row['relative']
        out.parent.mkdir(parents=True, exist_ok=True)
        normalized = normalize_video(Path(row['raw']), min_anchor_score=0.3)
        predicates = extract_predicates(normalized, min_score=0.3, window_size=5, motion_threshold=0.08)
        events = build_events(predicates, max_gap=2, min_confidence=0.3, composite_gap=4)
        graph = build_instance_graph(events)
        out.write_text(json.dumps(graph, ensure_ascii=False), encoding='utf-8')
        # Giữ một mẫu mỗi lớp để xem từng tầng pipeline.
        example = ARTIFACT_ROOT / split / row['label']
        if not example.exists():
            example.mkdir(parents=True)
            for name, obj in [('normalized', normalized), ('predicates', predicates), ('events', events)]:
                (example / f'{name}.json').write_text(json.dumps(obj, ensure_ascii=False, indent=2))
    ds = KeypointSequenceDataset(SPLIT_ROOT / f'{split}.txt', KEYPOINT_ROOT, split, 64,
                                cache_root=CACHE_ROOT, instance_graph_root=GRAPH_ROOT)
    for source, _ in ds.samples:
        assert ds.cache_path_for_source(source).is_file(), source
        assert ds.instance_graph_path_for_source(source).is_file(), source
    return ds

datasets = {s: make_graphs(s) for s in ('train', 'val')}
concept_rows = []
for row in samples[samples.split.isin(['train', 'val'])].to_dict('records'):
    graph = json.loads((GRAPH_ROOT / row['relative']).read_text())
    concepts = {(n['type'], n.get('properties', {}).get('name'))
                for n in graph['nodes'] if n.get('properties', {}).get('name')}
    for kind, name in concepts:
        concept_rows.append({'split': row['split'], 'label': row['label'], 'type': kind, 'concept': name})
concepts_df = pd.DataFrame(concept_rows, columns=['split','label','type','concept'])
coverage = concepts_df.groupby(['split','label','type','concept']).size().rename('clips').reset_index()
coverage.to_csv(RUN / 'concept_coverage.csv', index=False)
display(coverage.sort_values('clips', ascending=False).head(60))


## Xem clip và ghi chú người 3
Ưu tiên G07–G11; cũng xem mẫu của cả 13 lớp để viết template. Chỉ dùng train/val.
Nếu chưa có video trên Drive, nhận thư mục clips từ người 1. Xem JSON predicates/events trong `symbolic_examples` và `concept_coverage.csv`.


In [ ]:
REVIEW_LABEL = 'G07'
REVIEW_SPLIT = 'train'
assert REVIEW_SPLIT in ('train', 'val')
example_row = samples[(samples.label == REVIEW_LABEL) & (samples.split == REVIEW_SPLIT)].iloc[0]
video_base = DRIVE_ROOT / 'clips' / Path(example_row.relative).with_suffix('')
videos = [video_base.with_suffix(ext) for ext in ('.mp4', '.avi', '.mov')]
video = next((v for v in videos if v.exists()), None)
if video:
    preview = Path('/content') / (video.stem + '_preview.mp4')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(video),
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-an', str(preview)], check=True)
    display(Video(str(preview), embed=True))
else:
    print('Chưa có clip ở:', video_base)
display(coverage[(coverage.label == REVIEW_LABEL) & (coverage.split == REVIEW_SPLIT)])
notes_path = RUN / 'template_review_notes.csv'
if not notes_path.exists():
    pd.DataFrame({'label': LABELS, 'start_pose': '', 'motion': '', 'end_pose': '',
                  'confusable_with': '', 'train_val_examples': '', 'missing_signal': '',
                  'review_status': 'draft'}).to_csv(notes_path, index=False)
print('Ghi nhận xét vào:', notes_path)


## Template nháp 13 lớp — cần chỉnh bằng bằng chứng train/val
Dùng các concept mà extractor hiện có. B0A/G01/G08 và B0B/G02/G09 có thể thiếu tín hiệu đếm click; G07 chưa biểu diễn chắc chắn "mở hai lần". G10/G11 dùng biến đổi độ xòe ngón làm giả thuyết, phải đối chiếu video. Hướng trái/phải phụ thuộc quy ước mirror của người 1.
Không coi việc tạo đủ 13 graph là chứng minh chúng phân biệt được 13 lớp.
Cell này chỉ tạo YAML nếu chưa có, không ghi đè phần bạn đã chỉnh. Sửa YAML trên Drive rồi chạy lại cell build và matcher.


In [ ]:
TEMPLATE_YAML = RUN / 'gesture_templates.yaml'
draft = {
    'B0A': [('hand_shapes', 'index_extended')],
    'B0B': [('hand_shapes', 'index_middle_extended')],
    'G01': [('hand_shapes', 'index_extended'), ('finger_motions', 'index_finger_moves_down')],
    'G02': [('hand_shapes', 'index_middle_extended'), ('finger_motions', 'index_finger_moves_down')],
    'G03': [('motions', 'hand_moves_up')],
    'G04': [('motions', 'hand_moves_down')],
    'G05': [('motions', 'hand_moves_left')],
    'G06': [('motions', 'hand_moves_right')],
    'G07': [('hand_shapes', 'hand_open'), ('finger_motions', 'fingers_spread_increasing')],
    'G08': [('hand_shapes', 'index_extended'), ('finger_motions', 'index_finger_oscillates')],
    'G09': [('hand_shapes', 'index_middle_extended'), ('finger_motions', 'index_finger_oscillates')],
    'G10': [('finger_motions', 'fingers_spread_increasing')],
    'G11': [('finger_motions', 'fingers_spread_decreasing')],
}
if not TEMPLATE_YAML.exists():
    lines = ['version: 1', 'description: IPN draft - validate on train and val', 'gestures:']
    for label, requirements in draft.items():
        lines += [f'  {label}:', f'    description: Draft {label}', '    expected_events:', '      - subject: any']
        grouped = {}
        for field, name in requirements:
            grouped.setdefault(field, []).append(name)
        for field, names in grouped.items():
            lines += [f'        {field}:', '          required:']
            lines += [f'            - {name}' for name in names]
    TEMPLATE_YAML.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(TEMPLATE_YAML.read_text())


In [ ]:
from src_neurosymbolic.kg.build_template_graph import (
    parse_template_yaml, build_template_graph, DEFAULT_WEIGHTS, FIELD_TO_NODE_TYPE_AND_RELATION)
from src_neurosymbolic.kg.graph_matching import match_instance_to_templates

parsed = parse_template_yaml(TEMPLATE_YAML)
assert set(parsed['gestures']) == set(LABELS), 'YAML phải có đúng 13 nhãn.'
TEMPLATE_DIR = RUN / 'template_graphs'
TEMPLATE_DIR.mkdir(exist_ok=True)
graphs = []
train_concepts = concepts_df[concepts_df.split == 'train']
observed = set(zip(train_concepts['label'], train_concepts['type'], train_concepts['concept']))
unsupported = []
for label in LABELS:
    template = parsed['gestures'][label]
    assert template['expected_events'], f'Template rỗng: {label}'
    for event in template['expected_events']:
        assert event['requirements'], f'Event rỗng: {label}'
        for field, groups in event['requirements'].items():
            for level, names in groups.items():
                for name in names:
                    if level == 'required' and (label, FIELD_TO_NODE_TYPE_AND_RELATION[field][0], name) not in observed:
                        unsupported.append((label, field, name))
    graph = build_template_graph(label, LABEL_IDS[label], template, DEFAULT_WEIGHTS)
    graphs.append(graph)
    (TEMPLATE_DIR / f'{label}.json').write_text(json.dumps(graph, indent=2))
index = {'graph_type':'template_graph_index', 'template_count':13,
         'templates':[{'label_id':g['label_id'], 'label_name':g['label_name'],
                       'path':str(TEMPLATE_DIR / (g['label_name'] + '.json')), 'summary':g['summary']} for g in graphs]}
(TEMPLATE_DIR / 'index.json').write_text(json.dumps(index, indent=2))
assert {p.stem for p in TEMPLATE_DIR.glob('*.json')} == set(LABELS) | {'index'}
print('Required concept chưa xuất hiện trên train của đúng lớp:', unsupported)
matcher_rows = []
for row in tqdm(samples[samples.split.isin(['train','val'])].to_dict('records'), desc='Matcher'):
    result = match_instance_to_templates(json.loads((GRAPH_ROOT / row['relative']).read_text()), graphs, top_k=3)
    scores = result['scores']
    own = next(x for x in scores if x['label_id'] == row['label_id'])
    top_score = result['top_k'][0]['score']
    matcher_rows.append(dict(split=row['split'], label=row['label'], source=row['relative'],
                             predicted=result['prediction'], correct=result['prediction'] == row['label'],
                             own_score=own['score'], missing_required=own['missing_required_count'],
                             tied_best=sum(x['score'] == top_score for x in scores)))
matcher_df = pd.DataFrame(matcher_rows)
matcher_df.to_csv(RUN / 'matcher_train_val.csv', index=False)
display(matcher_df.groupby(['split','label'])[['correct','own_score','missing_required','tied_best']].mean())
print('Score hòa nhiều hoặc missing_required cao: xem lại template/tín hiệu trước khi nghiệm thu.')


## Train V4
Chạy smoke test một epoch trước; sau đó train đầy đủ chọn best bằng **validation accuracy**. Không truyền test-file cho trainer. Mặc định V4 train từ đầu; baseline người 2 chỉ dùng so sánh để tránh nhập nhầm checkpoint 27 lớp/khác kiến trúc.
Đặt `TEMPLATES_REVIEWED=True` sau khi đã xem clip, sửa YAML và ghi nhận giới hạn. Nếu Colab ngắt, dùng cùng RUN_NAME, RESUME=True; epochs là tổng số epoch mục tiêu, không phải số epoch cộng thêm. Dữ liệu trên Drive có thể làm I/O chậm.


In [ ]:
TEMPLATES_REVIEWED = False
EPOCHS = 30
BATCH_SIZE = 16
SEED = 42
assert TEMPLATES_REVIEWED, 'Hoàn tất review template train/val rồi đặt TEMPLATES_REVIEWED=True.'
review_notes = pd.read_csv(notes_path).fillna('')
assert set(review_notes.label) == set(LABELS) and len(review_notes) == 13
assert review_notes.review_status.eq('reviewed').all(), 'Ghi nhận xét 13 lớp rồi đổi review_status thành reviewed trong template_review_notes.csv.'
assert review_notes.train_val_examples.str.strip().ne('').all(), 'Ghi clip bằng chứng train/val cho từng lớp.'
assert not unsupported, f'Có concept chưa xuất hiện: {unsupported}. Kiểm tra/sửa template.'
def run_script(relative, args):
    command = [sys.executable, '-u', str(CODE / relative)] + list(map(str, args))
    print(' '.join(command))
    subprocess.run(command, cwd=REPO, check=True)

TRAIN_DIR = RUN / 'v4'
common = ['--keypoint-root', KEYPOINT_ROOT, '--cache-root', CACHE_ROOT,
          '--instance-graph-root', GRAPH_ROOT, '--template-dir', TEMPLATE_DIR,
          '--train-file', SPLIT_ROOT / 'train.txt', '--val-file', SPLIT_ROOT / 'val.txt',
          '--num-classes', 13, '--num-frames', 64, '--batch-size', BATCH_SIZE,
          '--num-workers', 0, '--device', DEVICE, '--seed', SEED, '--logit-branch', 'final']
run_script('src_neurosymbolic/train_cross_attention_v4.py',
           common + ['--output-dir', RUN / 'smoke', '--epochs', 1, '--limit-train', 26, '--limit-val', 13])


In [ ]:
# Lưu fingerprint để không resume trên template/split đã đổi.
signature = {'inputs': fingerprints, 'template_sha256': hashlib.sha256(TEMPLATE_YAML.read_bytes()).hexdigest(),
             'commit': commit, 'seed': SEED, 'batch_size': BATCH_SIZE,
             'data_sha256':data_digest.hexdigest(),
             'graph_parameters':{'min_anchor_score':0.3,'min_score':0.3,'window_size':5,'motion_threshold':0.08,'max_gap':2,'min_confidence':0.3,'composite_gap':4}}
signature_path = RUN / 'training_signature.json'
extra = []
if RESUME:
    assert signature_path.exists() and json.loads(signature_path.read_text()) == signature
    assert (TRAIN_DIR / 'last.pt').is_file()
    extra = ['--resume', TRAIN_DIR / 'last.pt']
else:
    assert not (TRAIN_DIR / 'last.pt').exists(), 'Đã có checkpoint: dùng RESUME hoặc RUN_NAME mới.'
    signature_path.write_text(json.dumps(signature, indent=2))
run_script('src_neurosymbolic/train_cross_attention_v4.py',
           common + ['--output-dir', TRAIN_DIR, '--epochs', EPOCHS] + extra)
assert (TRAIN_DIR / 'best.pt').is_file()
display(pd.DataFrame(json.loads((TRAIN_DIR / 'history.json').read_text())).tail())


## Test chính thức và baseline người 2
Chỉ bật sau khi chốt template/cấu hình/checkpoint bằng validation. Cung cấp checkpoint neural-only IPN 13 lớp và BASELINE_INPUT_HASHES của người 2 để có bảng so sánh độc lập. File hash có 4 khóa train.txt, val.txt, test.txt, manifest.csv; mỗi giá trị là SHA256 của file nguồn mà người 2 đã dùng để train. Không tạo hồ sơ thay cho người 2. Nhánh neural bên trong V4 đã được train tiếp, không phải baseline người 2.
Đây là so sánh hai hệ thống; delta không chứng minh riêng template gây cải thiện vì kiến trúc/ngân sách train có thể khác. Muốn kết luận về template, cần thêm ablation neural-only cùng kiến trúc và ngân sách. Notebook kiểm tra hash hồ sơ người 2 nhưng không thể tự chứng minh checkpoint thực sự đã được train đúng hồ sơ đó.


In [ ]:
RUN_FINAL_TEST = False
assert RUN_FINAL_TEST, 'Chốt thiết kế rồi đặt RUN_FINAL_TEST=True để chấm test.'
assert json.loads(signature_path.read_text()) == signature
assert hashlib.sha256(TEMPLATE_YAML.read_bytes()).hexdigest() == signature['template_sha256']
datasets['test'] = make_graphs('test')
run_script('src_neurosymbolic/test_cross_attention_v2.py', [
    '--checkpoint', TRAIN_DIR / 'best.pt', '--keypoint-root', KEYPOINT_ROOT,
    '--cache-root', CACHE_ROOT, '--instance-graph-root', GRAPH_ROOT,
    '--template-dir', TEMPLATE_DIR, '--test-file', SPLIT_ROOT / 'test.txt',
    '--split', 'test', '--batch-size', BATCH_SIZE, '--num-workers', 0,
    '--device', DEVICE, '--eval-branch', 'final', '--output', RUN / 'test_v4'])
if BASELINE_CHECKPOINT is not None:
    assert BASELINE_INPUT_HASHES is not None, 'Nhận input_hashes.json của người 2; không tự tạo thay hồ sơ baseline.'
    baseline_hashes = json.loads(Path(BASELINE_INPUT_HASHES).read_text())
    assert baseline_hashes == fingerprints, 'Baseline không cùng split/manifest. Thống nhất với người 2 trước khi so sánh.'
    baseline = torch.load(BASELINE_CHECKPOINT, map_location='cpu', weights_only=False)
    assert int(baseline['args']['num_classes']) == 13 and int(baseline['args']['num_frames']) == 64
    assert {int(k):v for k,v in baseline['label_names'].items()} == dict(enumerate(LABELS))
    # Đánh giá trực tiếp trên cùng dataset đã kiểm tra, tránh resolver khác của neural-only.
    sys.path.insert(0, str(CODE / 'neural_only_experiments'))
    from models import NeuralOnlyClassifier
    a = baseline['args']
    net = NeuralOnlyClassifier(**{k:a[k] for k in ('backbone','num_classes','num_frames','d_model','layers','heads','dropout')}).to(DEVICE)
    net.load_state_dict(baseline['model'])
    net.eval()
    from torch.utils.data import DataLoader
    loader = DataLoader(datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    baseline_rows = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Baseline người 2'):
            pred = net(batch['keypoints'].to(DEVICE)).argmax(1).cpu().tolist()
            baseline_rows.extend({'path':p, 'true_id':int(t), 'pred_id':int(y)}
                                 for p,t,y in zip(batch['path'], batch['label'], pred))
    pd.DataFrame(baseline_rows).to_csv(RUN / 'baseline_predictions.csv', index=False)
    (RUN / 'baseline_provenance.json').write_text(json.dumps(
        {'checkpoint':str(BASELINE_CHECKPOINT), 'sha256':hashlib.sha256(Path(BASELINE_CHECKPOINT).read_bytes()).hexdigest(),
         'args':a, 'epoch':baseline.get('epoch'), 'verified_input_hashes':baseline_hashes}, indent=2, default=str))
else:
    print('Chưa có baseline: chỉ xuất metric V4, chưa có kết luận hơn/kém người 2.')


## Metric cùng công thức, confusion matrix và bàn giao
Accuracy, macro-F1 và precision/recall/F1 cho đủ 13 lớp. Nếu có baseline, ghép bằng đường dẫn clip và xác nhận nhãn trùng trước khi so sánh. Báo delta theo lớp và những clip V4 sửa đúng/làm sai. Zip kết quả không chứa dataset tensor/video, nhưng có checkpoint nên có thể lớn.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
v4 = pd.read_csv(RUN / 'test_predictions.csv')
assert len(v4) == len(datasets['test']) and not v4.path.duplicated().any()
systems = {'V4': (v4.true_id, v4.final_pred_id)}
baseline_csv = RUN / 'baseline_predictions.csv'
if BASELINE_CHECKPOINT is not None and baseline_csv.exists():
    provenance = json.loads((RUN / 'baseline_provenance.json').read_text())
    assert provenance['sha256'] == hashlib.sha256(Path(BASELINE_CHECKPOINT).read_bytes()).hexdigest()
    assert provenance['verified_input_hashes'] == fingerprints
    b = pd.read_csv(baseline_csv)
    assert len(b) == len(v4) and set(b.path) == set(v4.path) and not b.path.duplicated().any()
    paired = v4.merge(b, on='path', suffixes=('_v4','_baseline'), validate='one_to_one')
    assert (paired.true_id_v4 == paired.true_id_baseline).all()
    systems['Baseline người 2'] = (paired.true_id_baseline, paired.pred_id)
    paired['v4_fixes_baseline'] = (paired.pred_id != paired.true_id_baseline) & (paired.final_pred_id == paired.true_id_v4)
    paired['v4_harms_baseline'] = (paired.pred_id == paired.true_id_baseline) & (paired.final_pred_id != paired.true_id_v4)
    paired.to_csv(RUN / 'paired_comparison.csv', index=False)
summaries, per_class = [], {}
for name, (truth, pred) in systems.items():
    report = classification_report(truth, pred, labels=list(range(13)), target_names=LABELS, output_dict=True, zero_division=0)
    table = pd.DataFrame(report).T
    slug = 'v4' if name == 'V4' else 'baseline'
    table.to_csv(RUN / f'{slug}_classification_report.csv')
    per_class[name] = table.loc[LABELS]
    summaries.append({'model':name, 'samples':len(truth), 'accuracy':accuracy_score(truth,pred),
                      'macro_f1':f1_score(truth,pred,labels=list(range(13)),average='macro',zero_division=0)})
    matrix = confusion_matrix(truth,pred,labels=list(range(13)))
    fig, ax = plt.subplots(figsize=(11,10))
    ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(name)
    fig.tight_layout()
    fig.savefig(RUN / f'{slug}_confusion.png', dpi=160)
    plt.show()
comparison = pd.DataFrame(summaries)
comparison.to_csv(RUN / 'comparison.csv', index=False)
display(comparison)
if len(per_class) == 2:
    delta = per_class['V4'][['precision','recall','f1-score']] - per_class['Baseline người 2'][['precision','recall','f1-score']]
    delta.to_csv(RUN / 'per_class_delta.csv')
    display(delta)
print('Bàn giao kèm template_review_notes.csv, giới hạn tín hiệu, input_hashes.json và checkpoint best.pt.')


In [ ]:
# Zip riêng kết quả, không đóng gói dataset nguồn.
archive = shutil.make_archive(str(RUN.parent / RUN.name), 'zip', root_dir=RUN)
print('Đã lưu trên Drive:', archive)
